In [44]:
from openai import OpenAI
import base64
import json

In [5]:
#pip install openai

In [6]:
data_path ="../../data/images/"

In [62]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="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",
)

In [63]:
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [64]:
image_path = r"C:\Users\jihed\OneDrive\Bureau\Thesis\MIAS_RAG\rag-llm-project\data\imagescaption\images\page180_img2.jpeg"  # replace with your local image
base64_image = encode_image_to_base64(image_path)

In [75]:
output_path ="../../data/images/generated_image_captions.json"

In [65]:
meta_data_path = "../../data/imagescaption/image_captions.json"
with open(meta_data_path, "r") as file:
    meta_data = json.load(file)

In [66]:
meta_data[0]

{'image_path': '../../data/imagescaption/images\\page95_img1.jpeg',
 'caption': 'Figure 3-1 Nutrient absorption in the gastrointestinal tract. SCFA, short-chain fatty acids.',
 'source_page': 95}

In [67]:
for item in meta_data:
    image_path = item["image_path"]
    caption = item["caption"]
    source_page = item["source_page"]

In [ ]:
def generate_caption(image_base64, tag):
    response = client.chat.completions.create(
        model="qwen/qwen2.5-vl-72b-instruct:free",
         messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"prepare a description of the following images? , this description will be used for generating a vector to be used in a rag system, so it should be concise and informative. The image is tagged with: {tag}."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_base64}"
                        }
                    }
                ]
            }
        ],
    )

    # Validate the response before extracting content
    if response and response.choices:
        return response.choices[0].message.content
    else:
        print(" No valid response received!")
        return None
   

In [73]:
"""def run_generation(meta_data):
    for idx, item in enumerate(meta_data):
        print(f"Processing item {idx + 1}/{len(meta_data)}")
        print("Item:", item)
        image_path = item["image_path"].replace("\\", "/")    # Ensure the path is in the correct format
        base64_image = encode_image_to_base64(image_path)
        tag = item["caption"]
        caption = generate_caption(base64_image, tag)
        print("Generated Caption:", caption  , "for image:", image_path.split("/")[-1])"""

'def run_generation(meta_data):\n    for idx, item in enumerate(meta_data):\n        print(f"Processing item {idx + 1}/{len(meta_data)}")\n        print("Item:", item)\n        image_path = item["image_path"].replace("\\", "/")    # Ensure the path is in the correct format\n        base64_image = encode_image_to_base64(image_path)\n        tag = item["caption"]\n        caption = generate_caption(base64_image, tag)\n        print("Generated Caption:", caption  , "for image:", image_path.split("/")[-1])'

In [76]:
def run_generation(meta_data, output_path="generated_image_captions.json"):
    results = []

    for idx, item in enumerate(meta_data):
        print(f"Processing item {idx + 1}/{len(meta_data)}")
        print("Item:", item)

        image_path = item["image_path"].replace("\\", "/")  # Normalize path
        try:
            base64_image = encode_image_to_base64(image_path)
            tag = item["caption"]
            caption = generate_caption(base64_image, tag)

            print("Generated Caption:", caption, "for image:", image_path.split("/")[-1])

            results.append({
                "image_path": image_path,
                "caption": caption,
                "original_tag": tag,
                "source_page": item.get("source_page", None),
                "info_source": "essentials-of-general-surgery-and-surgical-specialties Official Book"
            })

        except Exception as e:
            print(f"Failed to process {image_path}: {e}")

    # Save to JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"Captions saved to {output_path}")
    return results

In [77]:
run_generation(meta_data, output_path)

Processing item 1/475
Item: {'image_path': '../../data/imagescaption/images\\page95_img1.jpeg', 'caption': 'Figure 3-1 Nutrient absorption in the gastrointestinal tract. SCFA, short-chain fatty acids.', 'source_page': 95}
Generated Caption: The image illustrates nutrient absorption in the gastrointestinal tract, highlighting specific regions and the nutrients absorbed in each. The esophagus is shown but not involved in absorption. The stomach absorbs water, alcohol, copper, iodine, fluoride, molybdenum, and intrinsic factor. The duodenum absorbs calcium, phosphorus, magnesium, iron, copper, selenium, thiamin, riboflavin, niacin, biotin, folate, and vitamins A, D, E, K. The jejunum absorbs thiamin, riboflavin, niacin, pantothenate, biotin, folate, vitamin B6, vitamin C, vitamins A, D, E, K, calcium, phosphorus, magnesium, iron, zinc, chromium, manganese, molybdenum, and lipids. The ileum absorbs water, vitamin C, folate, vitamin B12, vitamin D, vitamin K, magnesium, and bile salts and a

[{'image_path': '../../data/imagescaption/images/page95_img1.jpeg',
  'caption': 'The image illustrates nutrient absorption in the gastrointestinal tract, highlighting specific regions and the nutrients absorbed in each. The esophagus is shown but not involved in absorption. The stomach absorbs water, alcohol, copper, iodine, fluoride, molybdenum, and intrinsic factor. The duodenum absorbs calcium, phosphorus, magnesium, iron, copper, selenium, thiamin, riboflavin, niacin, biotin, folate, and vitamins A, D, E, K. The jejunum absorbs thiamin, riboflavin, niacin, pantothenate, biotin, folate, vitamin B6, vitamin C, vitamins A, D, E, K, calcium, phosphorus, magnesium, iron, zinc, chromium, manganese, molybdenum, and lipids. The ileum absorbs water, vitamin C, folate, vitamin B12, vitamin D, vitamin K, magnesium, and bile salts and acids. The colon absorbs water, sodium, chloride, potassium, SCFAs, vitamin K, monosaccharides, amino acids, and di-, tripeptides.',
  'original_tag': 'Figure 3

In [25]:
completion = client.chat.completions.create(
  # model="qwen/qwen2.5-vl-72b-instruct:free",
  extra_body={},
  model="qwen/qwen2.5-vl-72b-instruct:free",
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "prepare a description of the following images? , this description will be used for generating a vector to be used in a rag system, so it should be concise and informative."
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
          }
        }
      ]
    }
  ]
)
print(completion.choices[0].message.content)

This image illustrates a medical procedure involving a cricothyrotomy. It shows a cross-sectional view of the throat with annotations highlighting the thyroid and cricoid cartilages. A scalpel handle is depicted entering the region between these cartilages, indicating the incision point for the procedure. The thyroid cartilage is superior and the cricoid cartilage is inferior in the tracheal structure. This is representative of Figure B.


In [9]:
completion.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Sure, please provide the images or a detailed description of the images so I can create concise and informative descriptions for them.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')

## Run model locally 

In [1]:
import json 
import base64
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

In [2]:
data_path ="../../data/images/"

In [3]:
meta_data_path = "../../data/imagescaption/image_captions.json"
with open(meta_data_path, "r") as file:
    meta_data = json.load(file)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
print(device)

cuda


In [6]:
# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto"
)
# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [7]:
print(f"Model is loaded on device: {device}")

Model is loaded on device: cuda


In [8]:
def generate_caption(image_base64, tag):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image":  f"data:image/jpeg;base64,{image_base64}",
                },
                {"type": "text", "text": f"prepare a description of the following images? , this description will be used for generating a vector to be used in a rag system, so it should be concise and informative.The image is tagged with: {tag}."},
            ],
        }
    ]

    # Preparation for inference
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(device)

    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text
   
   

In [9]:
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [10]:
import json
import os

def run_generation(meta_data, output_path="generated_image_captions.jsonl"):
    for idx, item in enumerate(meta_data):
        print(f"Processing item {idx + 1}/{len(meta_data)}")
        print("Item:", item)

        image_path = item["image_path"].replace("\\", "/")  # Normalize path
        try:
            base64_image = encode_image_to_base64(image_path)
            tag = item["caption"]
            caption = generate_caption(base64_image, tag)

            print("Generated Caption:", caption, "for image:", image_path.split("/")[-1])

            result = {
                "image_path": image_path,
                "caption": caption,
                "original_tag": tag,
                "source_page": item.get("source_page", None),
                "info_source": "essentials-of-general-surgery-and-surgical-specialties Official Book"
            }

            # Append to the file
            with open(output_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

        except Exception as e:
            print(f"Failed to process {image_path}: {e}")

    print(f"Captions saved to {output_path}")


In [11]:
output_path ="../../data/images/generated_image_captions_0-50.json"

In [12]:
run_generation(meta_data, output_path)

Processing item 1/475
Item: {'image_path': '../../data/imagescaption/images\\page95_img1.jpeg', 'caption': 'Figure 3-1 Nutrient absorption in the gastrointestinal tract. SCFA, short-chain fatty acids.', 'source_page': 95}
Generated Caption: ['The image is a labeled diagram of the human gastrointestinal (GI) tract, highlighting the regions where various nutrients are absorbed. The GI tract is divided into the esophagus, stomach, duodenum, jejunum, ileum, and colon. Each section is annotated with specific nutrients that are absorbed there:\n\n- **Esophagus**: Absorbs calcium, phosphorus, magnesium, iron, copper, selenium, thiamin, riboflavin, niacin, biotin, folate, vitamins A, D, E, and K.\n- **Stomach**: Absorbs water, ethanol, copper'] for image: page95_img1.jpeg
Processing item 2/475
Item: {'image_path': '../../data/imagescaption/images\\page109_img1.jpeg', 'caption': 'Figure 4-1 The Lethal Triad.', 'source_page': 109}
Generated Caption: ['The image is labeled "Figure 4-1 The Lethal 

KeyboardInterrupt: 